## Building A Chatbot

In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

• Conversational RAG: Enable a chatbot experience over an external source of data

• Agents: Build a chatbot that can take actions

This notebook will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# Loading Groq API 
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
# Loding LangSmith API for tracing
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Initializing the groq model

from langchain_groq import ChatGroq

model=ChatGroq(model="qwen/qwen3-32b")

model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x74eedcb66a50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x74eedcb67770>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hello, My name is Prashant and I am 4 years experience in Data Analyst domain.")])

AIMessage(content="<think>\nOkay, let me start by understanding the user's message. The user introduced themselves as Prashant with four years of experience as a Data Analyst. My first thought is to acknowledge their experience and offer assistance. Since they mentioned their role, I should consider what they might need help with. Common needs for a Data Analyst include technical questions, career advice, project ideas, or tools like SQL, Python, or data visualization.\n\nI need to respond in a friendly and encouraging manner. Maybe start by greeting them and expressing interest in their background. Then, ask how I can assist them today. It's important to keep the response open-ended to allow them to specify their needs. I should avoid making assumptions but provide a few examples of areas they might want help with, such as solving a problem, discussing tools, or career development. Keeping the tone supportive and approachable is key here.\n</think>\n\nHello Prashant! It's great to mee

In [4]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hello, My name is Prashant and I am 4 years experience in Data Analyst domain."),
        AIMessage(content="Hello Prashant! Thank you for sharing your background. With 4 years of experience \
                  in data analysis, you've likely worked with tools like SQL, Python, Excel, or data visualization \
                  platforms (e.g., Tableau, Power BI). How can I assist you today? For example:  \n\n- Are you looking\
                   for **career growth advice** (e.g., transitioning to data science, leadership roles)? \
                   \n- Do you need help with **technical challenges** (e.g., optimizing queries, cleaning data, building models)? \
                   \n- Are you seeking guidance on **learning new tools** (e.g., Python libraries like Pandas, machine learning, \
                  Big Data tools)?  \n- Or something else entirely?  \n\nLet me know your goals, and I’ll tailor my support! 😊'"),
        HumanMessage(content="Hey, What's my name and what I do?")
    ]
)

AIMessage(content='<think>\nOkay, let me start by recalling the user\'s previous message. They introduced themselves as Prashant with four years of experience in the Data Analyst field. Now, the user is asking, "Hey, What\'s my name and what I do?" So they want me to remember their name and role.\n\nFirst, I need to confirm that I can access the conversation history. Since this is part of the same interaction, I should definitely reference the previous details provided. The user might be testing if the AI can retain information from the conversation history or just wants a quick confirmation of their identity and role.\n\nNext, the user\'s current question is straightforward, but I should make sure to present the information clearly. They mentioned their name and experience, so I need to restate that accurately. Also, maybe they want to reinforce their identity for the AI, ensuring that future interactions are personalized.\n\nI should check if there\'s any deeper need here. Perhaps th

### Message History

We can use a message history class to wrap our model and make it stateful. This will keep the track of previous input and output of the model, and store them in some datastore. Future interaction will load those messages and pass them into the chain to as a part of conversation. Let's see how it would be implemented and work.

In [15]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

stored_id_table={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in stored_id_table:
        stored_id_table[session_id]=ChatMessageHistory()
    return stored_id_table[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [ ]:
# Configuration with respect to Conversation_1
config={"configurable":{"session_id":"chat1"}}

In [18]:
responses=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hello, My name is Prashant and I am 4 years experience in Data Analyst domain.")]},
    config=config,
)

In [19]:
responses.content

"<think>\nOkay, the user introduced himself as Prashant with 4 years of experience in Data Analyst. I need to acknowledge his introduction and offer assistance.\n\nFirst, I should thank him for sharing his details. Then, I should ask how I can help him today. Maybe he's looking for advice, resources, or career guidance. Since he has experience, perhaps he wants to upskill or move into a new role. I should keep the response friendly and open-ended to encourage him to specify his needs. Let me make sure the tone is positive and supportive. Also, use an emoji to keep it approachable. Alright, that should cover it.\n</think>\n\nHello Prashant! 😊  \nWelcome to the world of Data Analyst, and thank you for sharing your experience. With 4 years in the field, you’ve likely built a strong foundation. How can I assist you today? Whether it’s help with technical skills, career guidance, or exploring new tools/methodologies in data analysis, I’m here to help! Let me know what you’re working on or w

In [20]:
with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config,
)

AIMessage(content='<think>\nOkay, the user just asked "What is my name?" again. Let me check the conversation history.\n\nIn the first message, the user asked the same question, and I responded by saying I don\'t know unless they tell me. Then, in the next message, they introduced themselves as Prashant with 4 years of experience. So, I acknowledged their name then. Now, they\'re asking again.\n\nHmm, maybe they\'re testing if I remember, or perhaps they\'re concerned about privacy. But since they provided their name in the previous interaction, I should use that. I need to confirm their identity again and offer help. Also, since they mentioned their experience earlier, maybe they want to continue the conversation about their career or skills. \n\nI should respond by addressing them by name to show I remember, and ask how I can assist them further. Keep it friendly and open-ended. Let them know I\'m here to help with their specific needs. Make sure not to mention any personal info beyo

In [ ]:
# Changing the config --> session_id
# Configuration with respect to Conversation_2
config1={"configurable":{"session_id":"chat2"}}

responses1=with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config1,
)

In [23]:
responses1.content

'<think>\nOkay, the user is asking, "What is my name?" again. Let me check the history.\n\nIn the previous messages, the user introduced themselves as Prashant with 4 years of experience in Data Analyst. My last response confirmed their name as Prashant. Now, they\'re asking the same question again.\n\nHmm, why might they be asking again? Maybe they want to test if I remember, or perhaps they\'re in a conversation where they need to confirm their name again. Alternatively, could there be a technical issue where the session is resetting? But the conversation history is available, so that\'s unlikely.\n\nI need to make sure I acknowledge their name again. Since I already confirmed it, I should reiterate that I remember their name and offer further help. Also, maybe they want to switch the topic or need something else. I should keep the response friendly and open-ended to encourage them to ask for what they need next.\n\nLet me structure the response: Start with their name, mention their 

In [25]:
responses1=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hey my name is Amit.")]},
    config=config1,
)

In [26]:
responses1.content

'<think>\nOkay, the user introduced himself as Amit. I need to acknowledge his name and make sure to use it in the response. Since he shared his name, I should address him by that to make the conversation more personal and friendly.\n\nI should thank him for sharing his name to show appreciation. Then, I should offer further assistance in a welcoming manner. Maybe ask how I can help him now that I know his name. Keeping the tone positive and helpful is important. Let me make sure the response is concise and friendly.\n</think>\n\nHello, Amit! 😊 Nice to meet you! How can I assist you today?'

In [27]:
responses2=with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config1,
)
responses2.content

'<think>\nOkay, the user asked, "What is my name?" again. Let me check the history.\n\nIn the previous messages, the user first asked for their name, and I responded that I don\'t have access unless they share it. Then they said, "Hey my name is Amit," so I acknowledged it with a friendly greeting. Now they\'re asking again, "What is my name?"\n\nHmm, they might be testing if I remember their name. Since they provided it earlier, I should confirm that I do remember. But wait, do I have a way to retain that information beyond the current session? Probably not, since each interaction is stateless unless the system is designed to remember. But in this context, I should act as if I remember because they provided it in the same conversation thread.\n\nI need to make sure the response is accurate and friendly. Let me verify the history again. Yes, the user introduced themselves as Amit. So the correct response is to say Amit. I should also acknowledge that I remember to show that I\'m paying

### Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt|model

In [31]:
chain.invoke({"messages":[HumanMessage(content="Hello, My name is John.")]})

AIMessage(content='<think>\nOkay, the user introduced himself as John. I should acknowledge his name and offer assistance. Let me keep it friendly and open-ended.\n\nHello John! Nice to meet you. How can I assist you today? If you have any questions or need help with something, feel free to let me know!\n</think>\n\nHello John! Nice to meet you. How can I assist you today? If you have any questions or need help with something, feel free to let me know!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 37, 'total_tokens': 135, 'completion_time': 0.180134301, 'completion_tokens_details': None, 'prompt_time': 0.001758743, 'prompt_tokens_details': None, 'queue_time': 0.159322602, 'total_time': 0.181893044}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c9bbb-f93e-7980-a38e-64ee0660ceae-0', tool_calls=[]

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)


## Creating another config
config3={"configurable":{"session_id":"chat3"}}
query_response=with_message_history.invoke([HumanMessage(content="Hello, My name is John.")],
                                           config=config3)

query_response.content 

'<think>\nOkay, the user said, "Hello, My name is John." That\'s a greeting and an introduction. I need to respond appropriately. Let me start by acknowledging his greeting and welcoming him. Maybe say "Hello, John!" to be friendly. Then ask if there\'s something you can help him with. Keep it open-ended so he can decide what he needs. Make sure the tone is positive and helpful. Let me check for any typos or errors. Yep, looks good. Ready to respond.\n</think>\n\nHello, John! 😊 How can I assist you today? If you have any questions or need help with something, just let me know!'

In [33]:
# Adding some more complexity

prompt1=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant and answer all the questions to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain1=prompt1|model

In [35]:
chain1.invoke({"messages":[HumanMessage(content="Everything in the world is created by Nature.")],
              "language":"Hindi"}).content

'<think>\nOkay, the user said, "Everything in the world is created by Nature." Let me unpack that. First, I need to understand what they mean by "Nature." In many contexts, Nature refers to the physical world and its phenomena, as opposed to human-made or artificial things. So the statement is asserting that all things originate from natural processes.\n\nBut wait, in philosophical terms, some might argue that concepts or ideas aren\'t physical, so do they count as created by Nature? Also, there\'s the question of whether human activities are part of Nature or a separate category. For example, technology is a human creation, but humans themselves are part of Nature. So maybe the user is making a broad statement that all existence, including human creations, stems from natural laws and processes.\n\nI should also consider different perspectives. From a scientific standpoint, yes, everything is governed by natural laws, and all matter and energy are part of the universe, which evolved th

Let's now wrap this with a more complicated chain in a Message History class. This time, because there are multiple key in the input, we need to specify the correct key to use and same the chat history.

In [38]:
with_message_history=RunnableWithMessageHistory(
    chain1,
    get_session_history,
    input_messages_key="messages"
)

# Creating another config
config4={"configurable":{"session_id":"chat4"}}

def get_response(chain,history_func,input_messages_key:str,content:str,language_key:str,config):
    with_message_history=RunnableWithMessageHistory(
        chain,
        history_func,
        input_messages_key=input_messages_key
    )
    query_response=with_message_history.invoke(
        {input_messages_key: [HumanMessage(content=content)], "language":language_key},
        config=config4
    )

    return query_response.content

In [39]:
response=get_response(chain=chain1,
                      history_func=get_session_history,
                      input_messages_key='messages',
                      content="Hi, How are you?",
                      language_key="Spanish",
                      config=config4)

response

'<think>\nOkay, the user just said "Hi, How are you?" after introducing himself as Prashant. Let me check the history. The previous interaction was in Hindi, where I greeted him and offered help. Now he\'s asking how I am.\n\nI need to respond in Spanish as per the user\'s current request, but wait, the user might be switching languages. Wait, the user\'s instruction says to respond in Spanish, but his messages are in English. Hmm. Let me check the initial instruction again. The user wrote "Please respond in Spanish." So I should respond in Spanish regardless of the user\'s language. But the user\'s last message is in English. Maybe he\'s testing if I can switch languages.\n\nWait, the system message says: "You are a helpful assistant and answer all the questions to the best of your ability in Spanish." So regardless of the user\'s input language, I should respond in Spanish. So even if the user writes in English, I need to reply in Spanish. So the correct approach is to answer in Span

### Manage the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

### Trim Message

`trim_messages` in **LangChain** is a utility used to **limit the size of a conversation history** before sending it to an LLM.

It helps prevent:

* 🚫 Context window overflow
* 🚫 Token limit errors
* 🚫 Unnecessary token usage (cost savings)

---

##### 📌 Why `trim_messages` Is Needed

When building chatbots with memory, message history keeps growing:

```text
Human → AI → Human → AI → Human → AI ...
```

If you keep passing all previous messages to the model:

* You may exceed token limits
* Responses become slower
* Costs increase

So we **trim older messages intelligently**.

---

##### 🧠 What `trim_messages` Does

It:

* Counts tokens (or message length)
* Removes older messages
* Keeps important structure intact (like system message)
* Returns a shortened message list

---

##### 🎯 Common Pattern in Chatbots

Used inside a chain like this:

```python
from langchain_core.runnables import RunnableLambda

trimmer = RunnableLambda(
    lambda x: {
        "messages": trim_messages(
            x["messages"],
            max_tokens=1000,
            token_counter=model,
            strategy="last",
            include_system=True,
        )
    }
)

chain = trimmer | prompt | model
```

This ensures:

1. Messages are trimmed
2. Prompt is built
3. Model is called

---

##### 🚀 Summary

`trim_messages` is used to:

* Control memory size
* Stay within token limits
* Optimize cost & performance
* Maintain structured chat history

---


In [ ]:
from langchain_core.messages import trim_messages,SystemMessage

trimmer=trim_messages(
    max_tokens=100,
    start_on='human',
    include_system=True,
    strategy='last',
    token_counter=model,
    allow_partial=False
)

messages =[
    SystemMessage(content="you're a good assistant"), 
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"), 
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"), 
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
    ]

trimmer.invoke(messages)

/home/prashant/.local/lib/python3.13/site-packages/langchain_core/language_models/base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [42]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

new_chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt1
    | model
)

query_response=new_chain.invoke({"messages":messages +[HumanMessage(content="What ice-cream do I like?")],
                 "language":"English"})

query_response.content

'<think>\nOkay, the user asked, "What ice-cream do I like?" Let me check the conversation history.\n\nEarlier, the user mentioned, "I like vanilla ice cream." So they\'ve already stated their preference. I need to confirm if they\'re asking for a reminder or if they want to discuss it further. Since the question is straightforward, the answer should be vanilla. But maybe they want to explore other options or have a follow-up question. I should keep it friendly and offer help if they need more information. Just make sure the answer is clear and matches their previous input.\n</think>\n\nYou mentioned earlier that you like **vanilla ice cream**! 🍦 Is there anything else I can help you with?'

In [43]:
# Creating another config
config5={"configurable":{"session_id":"chat5"}}

# Lest Wrap this with Message History
response=get_response(chain=new_chain,
                      history_func=get_session_history,
                      input_messages_key='messages',
                      content="What ice-cream do I like??",
                      language_key="English",
                      config=config5)

response

'<think>\nOkay, the user is asking, "What ice-cream do I like??" I need to figure out how to respond. First, I realize that I don\'t have access to the user\'s personal preferences or data unless they\'ve shared it with me in this conversation. Since this is a new interaction, there\'s no prior information provided about their ice cream preferences.\n\nSo, the user might be testing if I can somehow know their preferences, but realistically, I can\'t. My role is to be helpful and informative. Therefore, the best approach is to explain that I can\'t know their specific likes but can provide general information about popular flavors or ask them to share their preferences so I can offer suggestions.\n\nI should also consider that the user might be looking for guidance on how to discover their own preferences or looking for recommendations. In that case, I can ask follow-up questions to narrow down possible flavors based on their tastes. For example, if they like sweet, fruity, or creamy te